# 🔬 MLens Tutorial — Explainable ML Audit Tool

Welcome! This notebook walks through every feature of **MLens** end-to-end:

1. Train a model
2. Run a full audit (SHAP + Fairness + Drift)
3. Interpret the results
4. Export reports (HTML / PDF)
5. Detect concept drift over time
6. Log to MLflow / W&B

> 📦 GitHub: [github.com/saiganesh47/mlens](https://github.com/saiganesh47/mlens)

## 1. Setup

In [ ]:
!pip install -q mlens scikit-learn pandas numpy

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from mlens import ModelAuditor

print("✅ MLens imported successfully")

## 2. Load Data — UCI Adult Income Dataset

We'll use the classic Adult Income dataset — predicting whether someone earns >$50K/year.
This dataset is famous for having **gender bias** baked in, making it perfect for a fairness audit demo.

In [ ]:
dataset = fetch_openml("adult", version=2, as_frame=True, parser="auto")
df = dataset.frame.dropna()

target_col    = "class"
sensitive_col = "sex"

# Encode categoricals
le = LabelEncoder()
df[target_col] = le.fit_transform(df[target_col])

cat_cols = df.select_dtypes("category").columns.tolist()
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

X = df.drop(columns=[target_col, sensitive_col])
y = df[target_col]
sensitive = df[sensitive_col]

print(f"Dataset shape: {X.shape}")
df.head()

## 3. Train / Test Split

In [ ]:
(X_train, X_test,
 y_train, y_test,
 s_train, s_test) = train_test_split(
    X, y, sensitive,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")

## 4. Train a Model

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
)
model.fit(X_train, y_train)

acc = model.score(X_test, y_test)
print(f"✅ Test accuracy: {acc:.4f}")

## 5. Run the Full MLens Audit 🔬

This is where the magic happens. One object, one method call —
you get explainability, fairness, and drift diagnostics all at once.

In [ ]:
auditor = ModelAuditor(
    model=model,
    X_train=X_train,
    X_test=X_test,
    y_test=y_test,
    sensitive_features=s_test,
    feature_names=list(X.columns),
    model_name="GBT-AdultIncome",
    shap_background_samples=100,
)

report = auditor.run()

## 6. Read the Executive Summary

In [ ]:
print("── Audit Summary ──────────────────────────────")
for line in report.summary_lines:
    print(f"  {line}")

## 7. SHAP — Why Did the Model Decide That?

SHAP (SHapley Additive exPlanations) tells you which features drove each prediction.
MLens auto-selects the fastest explainer for your model type.

In [ ]:
print("── Top 10 Features (SHAP) ─────────────────────")
for f in report.shap_result.top_features(n=10):
    bar = "█" * int(f["mean_abs_shap"] * 100)
    print(f"  {f['rank']:>2}. {f['name']:<20} {bar:<25} {f['mean_abs_shap']:.4f}")

In [ ]:
# Explain a single prediction
sample_idx = 0
local = report.shap_result.local_explanation(sample_idx)

print(f"── Why was sample #{sample_idx} predicted this way? ──")
for item in local[:5]:
    direction = "↑ increases" if item["shap_value"] > 0 else "↓ decreases"
    print(f"  {item['feature']:<20} {direction} prediction by {abs(item['shap_value']):.4f}")

## 8. Fairness — Is the Model Biased?

This dataset is known to have a gender gap in outcomes.
MLens flags it automatically using EEOC and equalized-odds thresholds.

In [ ]:
fr = report.fairness_result

print("── Fairness Metrics ───────────────────────────")
print(f"  Demographic Parity Gap : {fr.demographic_parity_gap:.4f}")
print(f"  Equalized Odds Gap     : {fr.equalized_odds_gap:.4f}")
print(f"  Disparate Impact       : {fr.disparate_impact:.4f}")
print(f"  Overall Fair?          : {'✅ Yes' if fr.is_fair else '⚠️ No'}")

if fr.flags:
    print("\n  Flags:")
    for flag in fr.flags:
        print(f"  ⚠️  {flag}")

In [ ]:
# Per-group breakdown as a DataFrame
pd.DataFrame(fr.per_group_metrics)

## 9. Drift — Has the Data Changed?

Drift detection compares your training distribution against
the test/production distribution using PSI and KS-test.

In [ ]:
dr = report.drift_result

print(f"── Drift Status: {dr.overall_status.upper()} ──")
drifted = dr.drifted_features()
if drifted:
    print(f"Drifted features: {', '.join(drifted)}")
else:
    print("No features drifted.")

pd.DataFrame(dr.feature_results)

## 10. Export Reports

In [ ]:
# HTML report (interactive, offline-viewable)
report.save("mlens_audit_report.html")

# PDF report (for stakeholders / compliance docs)
from mlens.report.pdf_generator import PDFGenerator
PDFGenerator(report).render("mlens_audit_report.pdf")

print("✅ Reports saved: mlens_audit_report.html, mlens_audit_report.pdf")

## 11. Bonus — Concept Drift Over Time 📈

Feature drift checks the *inputs*. Concept drift checks whether the
model's *error rate* is changing over time — a sign it needs retraining.

In [ ]:
from mlens.drift.concept_drift import ConceptDriftDetector

y_pred = model.predict(X_test)

detector = ConceptDriftDetector(method="adwin")
drift_result = detector.detect(y_test.values, y_pred)

print(drift_result.summary)

## 12. Bonus — Log to MLflow 🔗

Track every audit as an MLflow run for full experiment history.

In [ ]:
# Uncomment to log to MLflow:
# from mlens.integrations.mlflow_tracker import MLflowTracker
# tracker = MLflowTracker(experiment_name="mlens-tutorial")
# run_id = tracker.log(report)
# print(f"Logged to MLflow run: {run_id}")

## 🎉 You're Done!

You've now run a complete production-grade ML audit covering:
- ✅ Explainability (SHAP)
- ✅ Fairness (Demographic Parity, Equalized Odds, Disparate Impact)
- ✅ Feature Drift (PSI + KS-test)
- ✅ Concept Drift (ADWIN)
- ✅ Exportable HTML/PDF reports

### Next Steps
- Try MLens on your own model: `ModelAuditor(model, X_train, X_test, y_test)`
- Run the CLI: `mlens audit model.pkl X_test.csv y_test.csv`
- Launch the dashboard: `streamlit run dashboard/app.py`
- Deploy the API: `docker-compose up --build`

⭐ Star the repo: [github.com/saiganesh47/mlens](https://github.com/saiganesh47/mlens)